##Install Dependencies

In [3]:
!pip install groq -q

##MoE Code

In [4]:
import getpass
import os
from groq import Groq

# 1. API Setup
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# 2. Define Your Experts (Added 'tool' expert)
MODEL_CONFIG = {
    "technical": "You are a Technical Support Expert. Your responses should be rigorous, code-focused, and precise.",
    "billing": "You are a Billing Support Expert. Your responses should be empathetic, financial-focused, and policy-driven.",
    "general": "You are a friendly General Support Assistant. Help users with casual chat.",
    "tool": "You are a helpful Data Assistant. You will be provided with raw JSON data from a tool. Use that data to answer the user's question naturally."
}

# 3. The Mock Tool Function
def get_bitcoin_price():
    """Mock function to simulate an API call fetching crypto prices."""
    print("   🔧 [System]: Executing tool 'get_bitcoin_price' to fetch live data...")
    return '{"currency": "BTC", "price": "$68,420.50", "trend": "upward"}'

# 4. The Router Function
def route_prompt(user_input):
    # We added 'tool' to the list and gave it instructions on when to trigger it
    routing_system_prompt = """
    Classify this text into exactly one of these categories: [technical, billing, general, tool].
    If the user asks for current prices, live data, or Bitcoin, classify it as 'tool'.
    Return ONLY the word, nothing else. No punctuation, no explanation, no markdown.
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": routing_system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0,
        max_tokens=10
    )

    category = response.choices[0].message.content.strip().lower()
    if category not in MODEL_CONFIG: return "general"
    return category

# 5. The Orchestrator
def process_request(user_input):
    category = route_prompt(user_input)
    print(f"🚦 [Router Output]: Request routed to '{category.upper()}' expert.")

    system_prompt = MODEL_CONFIG[category]

    # --- BONUS CHALLENGE LOGIC ---
    if category == "tool":
        # Intercept the request and call our python function
        tool_data = get_bitcoin_price()
        # Augment the prompt with the injected data
        user_input = f"User Question: {user_input}\n\nLive Tool Data: {tool_data}"
    # -----------------------------

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    return response.choices[0].message.content

Enter your Groq API Key: ··········


##Test Your Router

In [5]:
# Test 1: Technical Issue
print("--- TEST 1 ---")
answer_1 = process_request("My python script is throwing an IndexError on line 5.")
print(f"AI: {answer_1}\n")

# Test 2: Billing Issue
print("--- TEST 2 ---")
answer_2 = process_request("I was charged twice for my subscription this month.")
print(f"AI: {answer_2}\n")

# Test 3: Bonus Challenge - Tool Use!
print("--- TEST 3 (BONUS) ---")
user_query_3 = "What is the current price of Bitcoin?"
print(f"User: {user_query_3}")
answer_3 = process_request(user_query_3)
print(f"AI: {answer_3}\n")

--- TEST 1 ---
🚦 [Router Output]: Request routed to 'TECHNICAL' expert.
AI: To troubleshoot the `IndexError`, I'll need more information about your script. Please provide the following:

1. The exact error message you're seeing, including the line number and any relevant error text.
2. The code snippet that's causing the error, starting from the beginning of the line that triggers the error and including any relevant surrounding code.
3. The version of Python you're running.

Without this information, it's difficult to provide a specific solution.

In general, an `IndexError` occurs when you're trying to access an element in a list or other sequence that doesn't exist. For example:

```python
my_list = [1, 2, 3]
print(my_list[3])  # IndexError: list index out of range
```

To fix this issue, you'll need to ensure that the index you're using is within the valid range for the list or sequence.

Here are some common mistakes that can cause an `IndexError`:

* Accessing an index that's out